# ISOM 835 · Session 8 — Gradient Boosting
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Nov 9 · Prof. Hasan Arslan**

**Tonight in one sentence:** last week 300 managers **voted**. Tonight we build a **team of trainees**, where each new trainee studies only **the mistakes the team is still making**.

**How to use this notebook:** run the cells top to bottom (Shift + Enter). Read the short text above each cell first: it tells you **what the cell does** and **how to read the output**. You do not need to understand every line of code tonight. You do need to understand every *result*. (The first code cell only checks your Python setup: run it and move on.)

🎮 **Play along:** the [Gradient Boosting Explorer](https://isom-835.vercel.app/explore/boosting/) lets you add trees one at a time and watch the model learn from its mistakes. It works on your phone.

### Tonight's map

| Step | The question it answers |
|---|---|
| **0** | Same hotel, same exam: what is the score to beat? |
| **1** | What does "learning from mistakes" look like? (one clue, one picture) |
| **2** | Can we build boosting ourselves, by hand? (yes, in six lines) |
| **3** | What is the real tool, and is it better than last week's forest? |
| **4** | Can a boosted model have *too many* trees? (yes: this is new) |
| **5** | How does it know when to stop? (early stopping) |
| **6** | Big steps or small steps? (the learning rate) |
| **7** | Who are XGBoost and LightGBM? |
| **8** | What is new in 2026? (models that have "read" millions of tables) |
| **9** | The leaderboard, and your turn |

Look for three kinds of boxes: **How to read it** (what an output means), **Stop and notice** (the lesson), and **🧠 Quick check** (a 20-second question to test yourself; click to see the answer).

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

## The business situation (same as last week)

You are still the **revenue manager** for two hotels in Portugal. **About 37% of bookings get canceled**, and for each booking you want one number: **how likely is it to be canceled?**

Last week's story, in three lines:

- **One decision tree** is a flowchart of yes/no questions. Easy to read, but **fragile**. Best single tree: **0.916**.
- A **random forest** is a crowd of 300 trees that **vote**. Their different mistakes cancel out. Score: **0.942**.
- The trees in a forest **never talk to each other**. Each one works alone.

Tonight's idea: **what if each tree could see what the others got wrong?**

| | 🌲 Random forest (last week) | 🚀 Gradient boosting (tonight) |
|---|---|---|
| The picture | a **committee**: 300 managers vote | a **relay team**: each trainee fixes what is left |
| How trees are built | all at once, independently | **one after another**, each looking at the team's mistakes |
| Each tree is… | big (a memorizer) | **small** (a few questions) |
| How they combine | **average** the votes | **add up** the corrections |

The next cell loads the same bookings and rebuilds the same train/test split as last week, so every score tonight can be compared with last week's.

In [ ]:
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Load the bookings (the competition's training data, hosted in our course repo)
URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/hotel_bookings.csv'
raw = pd.read_csv(URL)
LEAKS = ['reservation_status', 'reservation_status_date', 'assigned_room_type']   # last week: these columns are cheating

bookings = raw.sample(30_000, random_state=835)            # the same random 30,000 bookings as last week
y = bookings['is_canceled']                                 # target: 1 = canceled
X_raw = bookings.drop(columns=['is_canceled'] + LEAKS)      # inputs: everything else, minus the leaks
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(X_raw, y, test_size=0.2, stratify=y, random_state=835)

def tree_prep(frame):
    # The prep station for trees (same as last week): fill blanks, turn text into code numbers. No scaling, no one-hot.
    numbers = frame.select_dtypes(include='number').columns.tolist()
    text = [c for c in frame.columns if c not in numbers]
    return ColumnTransformer([
        ('numbers', SimpleImputer(strategy='constant', fill_value=-1), numbers),
        ('text', Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing')),
                           ('codes', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), text),
    ], verbose_feature_names_out=False).set_output(transform='pandas')

prep = tree_prep(X_raw)
X_tr = prep.fit_transform(X_tr_raw)     # learn the dictionary on the training bookings only
X_te = prep.transform(X_te_raw)         # apply the same dictionary to the test bookings

print(f'{len(raw):,} bookings in the file · tonight: {len(X_tr):,} to train, {len(X_te):,} held back as the exam · {y_tr.mean():.0%} canceled')

## 0. The score to beat

Same grading as last week: **AUC**. *If I pick one booking that canceled and one that didn't, how often does the model give the canceler the higher score?* **0.5** = a coin flip, **1.0** = perfect.

The next cell rebuilds last week's two best models and puts them on tonight's **leaderboard**.

In [ ]:
leaderboard = []   # every model's test AUC goes here

def score(name, model=None, scores=None):
    # Grade a model (or a list of scores) on the test bookings and add it to the leaderboard
    if scores is None:
        scores = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, scores)
    leaderboard.append({'model': name, 'test AUC': auc})
    print(f'{name:45s} test AUC {auc:.3f}')

best_tree = DecisionTreeClassifier(min_samples_leaf=50, random_state=835).fit(X_tr, y_tr)
forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=835).fit(X_tr, y_tr)

score('Last week: best single tree', best_tree)
score('Last week: random forest, 300 trees', forest)

**Stop and notice:** the forest's **0.942** is the score to beat tonight.

## 1. Learning from mistakes: one clue, one picture

Think about how you study for an exam.

- **The forest way:** take 300 practice exams, each time starting from zero, and average your answers.
- **The boosting way:** take a practice exam, **look at what you got wrong**, and study only that. Repeat.

To *see* this, we start with only **one clue**: `lead_time` (how many days ahead the guest booked). Here is the recipe:

1. **Round 0:** give every booking the same score: the overall cancel rate, **37%**.
2. Compute each booking's **miss**: *what really happened* minus *our score*. (A guest who canceled is a 1, so the miss is 1 − 0.37 = **+0.63**: we were too low. A guest who stayed is a 0, so the miss is 0 − 0.37 = **−0.37**: we were too high.)
3. A **trainee** (a tiny tree that may ask just **one question**) learns to predict the **miss**.
4. Add **a part** of the trainee's correction to our scores. Go back to step 2.

The next cell runs that recipe and takes a snapshot after 1, 3, 10 and 100 trainees.

In [ ]:
lead_tr = X_tr[['lead_time']]                                  # one clue only
grid = pd.DataFrame({'lead_time': np.arange(0, 366)})          # every lead time from 0 to 365 days, for drawing
STEP = 0.3                                                     # how much of each correction we keep (more on this in Step 6)

# The truth we are chasing: the real cancel rate at each lead time (training bookings, in 25 groups)
groups = pd.qcut(lead_tr['lead_time'], 25, duplicates='drop')
truth = pd.DataFrame({'lead_time': lead_tr['lead_time'], 'canceled': y_tr}).groupby(groups, observed=True).mean()

team_scores = np.full(len(y_tr), y_tr.mean())                  # round 0: everyone gets 37%
team_curve = np.full(len(grid), y_tr.mean())

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
snapshots = {1: 0, 3: 1, 10: 2, 100: 3}
for round_number in range(1, 101):
    miss = y_tr - team_scores                                                  # what the team still gets wrong
    trainee = DecisionTreeRegressor(max_depth=1).fit(lead_tr, miss)            # a tiny tree: ONE question about lead time
    team_scores += STEP * trainee.predict(lead_tr)                             # add part of its correction
    team_curve += STEP * trainee.predict(grid)
    if round_number in snapshots:
        ax = axes[snapshots[round_number]]
        ax.scatter(truth['lead_time'], truth['canceled'], color='#d64545', s=28, label='what really happened')
        ax.plot(grid['lead_time'], team_curve, color='#5b6bd8', lw=2.5, label="the team's score")
        ax.set_title(f'after {round_number} trainee' + ('s' if round_number > 1 else ''))
        ax.set_xlabel('lead time (days booked ahead)')
axes[0].set_ylabel('chance of cancellation'); axes[0].legend(loc='lower right', fontsize=9)
plt.suptitle('Boosting with one clue: each trainee fixes a little of what is still wrong', y=1.04, fontsize=13); plt.show()

**How to read it:** the **red dots** are the truth: the real cancel rate for guests who booked that many days ahead. The **blue line** is the team's score. Left to right, the team gets more trainees.

**Stop and notice:**
- **After 1 trainee:** one question (*"booked more than 25 days ahead?"*) makes **one step**. Crude, but better than a flat 37% for everyone.
- **After 3 and 10:** each new trainee adds **another small step exactly where the team is still wrong**.
- **After 100:** a staircase that hugs the red dots. No single trainee is smart. **The team is.**

That is the whole idea of **boosting**: many **weak learners** (tiny trees), added one after another, each one correcting what is left.

🎮 Try it yourself: in the [Gradient Boosting Explorer](https://isom-835.vercel.app/explore/boosting/) press **Add 1 tree** a few times. The orange line is the next trainee's correction.

### 🧠 Quick check

After round 0 every booking has a score of 0.37. Guest A **canceled**. Guest B **stayed**. What is each guest's miss, and which way will the first trainee push each score?

<details><summary>Click for the answer</summary>

**Guest A:** 1 − 0.37 = **+0.63**. We were too low, so the trainee pushes A's score **up**.
**Guest B:** 0 − 0.37 = **−0.37**. We were too high, so the trainee pushes B's score **down**.
The miss has a size *and* a direction. (Statisticians call the miss a **residual**. You met residuals in Session 6.)
</details>

## 2. Boosting by hand, with all the clues

Now the same recipe with **all 28 columns**, and trainees that may ask **three questions** each (`max_depth=3`). We do the first three rounds slowly and watch **six real guests**.

In [ ]:
team_tr = np.full(len(y_tr), y_tr.mean())      # the team's score for every training booking (round 0: 37% for everyone)
team_te = np.full(len(y_te), y_tr.mean())      # ...and for every test booking

six = list(np.where(y_tr == 1)[0][:3]) + list(np.where(y_tr == 0)[0][:3])      # six guests to watch: three who canceled, three who stayed
watch = X_tr_raw.iloc[six][['lead_time', 'deposit_type', 'previous_cancellations']].copy()
watch['canceled?'] = y_tr.iloc[six].map({1: 'YES', 0: 'no'})
watch['round 0'] = team_tr[six]

for round_number in [1, 2, 3]:
    miss = y_tr - team_tr                                                          # 1. what the team still gets wrong
    trainee = DecisionTreeRegressor(max_depth=3, random_state=835).fit(X_tr, miss)  # 2. a small tree learns to predict the miss
    team_tr += STEP * trainee.predict(X_tr)                                         # 3. add 30% of its correction
    team_te += STEP * trainee.predict(X_te)
    watch[f'round {round_number}'] = team_tr[six]
    print(f'after trainee {round_number}: test AUC {roc_auc_score(y_te, team_te):.3f}')

watch.round(2)

**How to read it:** each row is one real guest. `round 0` is the starting score (0.37 for everyone). Then read **left to right**: guests who **canceled** should climb toward 1, guests who **stayed** should sink toward 0.

**Stop and notice:**
- Nobody jumps straight to the right answer. Each trainee moves the scores **a little**, and mostly **in the right direction**.
- Not every step is right for every guest (look for a score that wobbles). That's fine: the next trainee sees that miss and corrects it.
- The test AUC climbs with every trainee, and we have only used **three small trees**.

Three trainees is a small team. The next cell keeps going to **100**. It is the same three lines inside the loop: **miss, trainee, add**.

In [ ]:
rounds_train, rounds_test = [roc_auc_score(y_tr, team_tr)], [roc_auc_score(y_te, team_te)]     # scores after 3 trainees
for round_number in range(4, 101):
    miss = y_tr - team_tr
    trainee = DecisionTreeRegressor(max_depth=3, random_state=835).fit(X_tr, miss)
    team_tr += STEP * trainee.predict(X_tr)
    team_te += STEP * trainee.predict(X_te)
    rounds_train.append(roc_auc_score(y_tr, team_tr)); rounds_test.append(roc_auc_score(y_te, team_te))

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(range(3, 101), rounds_test, color='#d64545', lw=2.5, label='our hand-built team (test bookings)')
ax.axhline(leaderboard[0]['test AUC'], color='#999999', ls='--', label='best single tree (0.916)')
ax.axhline(leaderboard[1]['test AUC'], color='#2e8b57', ls='--', label='random forest (0.942)')
ax.set_xlabel('number of trainees (small trees) on the team'); ax.set_ylabel('test AUC')
ax.set_title('Boosting by hand: every trainee helps a little'); ax.legend(loc='lower right'); plt.show()

score('Boosting by hand, 100 small trees', scores=team_te)

**Stop and notice:**
- At **25 trainees**, our hand-built team **passes the best single tree** from last week. Each of our trees asks only 3 questions; that tree had about 270 leaves.
- With 100 trainees we reach about **0.93**, close to the forest, with six lines of code that we wrote ourselves.

**You just built gradient boosting.** That is really all it is: **miss, trainee, add, repeat**.

### Why is it called *gradient* boosting?

Imagine walking down a hill in thick fog. You can't see the bottom, but you can feel **which way is downhill** under your feet. So you take **a small step that way**, and feel again.

- The **hill** is the model's total error.
- The **miss** tells every booking's score **which way is downhill** (up for cancelers, down for stayers). In math, that direction is called the **gradient**.
- Each trainee is **one small step downhill**.

It is the same idea that trains **neural networks** (last week's bridge): *gradient descent*. A network nudges its **weights** downhill. Boosting adds a **new small tree** for each step.

One honest detail: our by-hand scores are plain numbers, so some of them drift below 0 or above 1. The real tool, next, fixes that the same way logistic regression did in Session 4: it works on a scale that always turns back into a chance between 0 and 1.

## 3. The real tool: `HistGradientBoostingClassifier`

Nobody boosts by hand at work. scikit-learn's tool is called **`HistGradientBoostingClassifier`**. A long name, three parts:

- **Gradient Boosting**: exactly what we just built.
- **Hist** (histogram): before it starts, it sorts every number column into at most 255 **buckets** (like age groups instead of exact ages). Trees then try 255 cut points instead of thousands. That makes it **very fast**, even on millions of rows.
- **Classifier**: it predicts yes/no, as a probability.

Two more gifts: it **handles blanks by itself** (no filling needed), and, like every tree model, it needs **no scaling and no one-hot**.

Same train bookings, same exam. We only change the model.

In [ ]:
start = time.time()
boost = HistGradientBoostingClassifier(random_state=835)      # all default settings: 100 small trees, learning rate 0.1
boost.fit(X_tr, y_tr)
print(f'trained {boost.n_iter_} trees in {time.time() - start:.1f} seconds\n')

score('Gradient boosting, default settings', boost)

**Stop and notice:** with **no tuning at all** and about a second of training, boosting **ties the forest** (0.942). The forest needed 300 big trees; boosting used 100 small ones.

Can it do better? Boosting has dials the forest doesn't have. The first one will surprise you.

## 4. Too many trees? This is new

Last week's rule for forests: **more trees never hurt**. Extra voters only make the vote steadier.

Boosting is different. Every new trainee is told: *"find something the team still gets wrong on the **training** bookings, and fix it."* At first, what's wrong is real patterns. Later, the only things left to fix are **quirks of individual bookings**. The team starts to **memorize**.

The next cell trains 1,000 trees and grades the team after every single tree, on the practice test (training bookings) and on the real exam (test bookings).

In [ ]:
long_run = HistGradientBoostingClassifier(max_iter=1000, early_stopping=False, random_state=835).fit(X_tr, y_tr)   # force all 1,000 trees

# staged_predict_proba replays the team's scores after 1 tree, 2 trees, 3 trees...
practice = [roc_auc_score(y_tr, p[:, 1]) for p in long_run.staged_predict_proba(X_tr)]
exam = [roc_auc_score(y_te, p[:, 1]) for p in long_run.staged_predict_proba(X_te)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 1001), practice, color='#5b6bd8', lw=2, label='practice test (training bookings)')
ax.plot(range(1, 1001), exam, color='#d64545', lw=2, label='real exam (test bookings)')
ax.set_xlabel('number of trees on the team'); ax.set_ylabel('AUC'); ax.set_ylim(0.88, 1.005)
ax.set_title('More trees: practice goes to perfect, the exam stops improving'); ax.legend(loc='lower right'); plt.show()

for n in [50, 100, 300, 500, 1000]:
    print(f'{n:5d} trees   practice {practice[n - 1]:.3f}   exam {exam[n - 1]:.3f}')

**Stop and notice:**
- **Practice (blue) climbs to a perfect 1.00.** The memorizer from Sessions 6 and 7 is back.
- **The exam (red) stops improving after about 300 trees** (≈ 0.947) and then just sits there. The last 700 trees learned **nothing useful**. They only cost time.
- It gets worse if you look closer: with too many trees the model becomes **overconfident**. It starts saying "99% sure" about bookings it has simply memorized. If you use those probabilities to make decisions (Session 5), that hurts.

So the number of trees is now a **dial that matters**. Too few: the team is still learning. Too many: it memorizes. How do we find the right number without peeking at the exam?

### 🧠 Quick check

Your colleague trained a boosted model with 5,000 trees and says: *"More trees never hurt. We learned that last week."* What do you tell them?

<details><summary>Click for the answer</summary>

**That rule is for forests.** In a forest, trees vote independently, so extra trees only steady the vote. In boosting, each new tree keeps **fixing leftover mistakes on the training data**, so after a point it is fitting quirks, not patterns. Boosting needs a **stopping rule**. That is Step 5.
</details>

## 5. Early stopping: the automatic brake

Think of a student preparing for a final. A good coach says: *"Keep studying while your **mock exam** scores improve. When they stop improving, **stop**. More cramming is just memorizing the practice questions."*

**Early stopping** is that coach:

1. Set aside a slice of the **training** bookings as a **mock exam** (here 15%). The model never trains on them.
2. After each new tree, grade the team on the mock exam.
3. If the mock score has not improved for **50 trees in a row**, **stop**.

The real exam (our test bookings) stays untouched. We allow up to 3,000 trees and let the coach decide.

In [ ]:
start = time.time()
boost_es = HistGradientBoostingClassifier(
    max_iter=3000,               # allow up to 3,000 trees...
    early_stopping=True,         # ...but let the coach stop us
    validation_fraction=0.15,    # the mock exam: 15% of the training bookings
    n_iter_no_change=50,         # stop when 50 trees in a row bring no improvement
    random_state=835)
boost_es.fit(X_tr, y_tr)

print(f'allowed 3,000 trees · the coach stopped at {boost_es.n_iter_} · {time.time() - start:.1f} seconds\n')
score('Gradient boosting + early stopping', boost_es)

mock_error = -np.array(boost_es.validation_score_)      # the mock-exam error after each tree (lower is better)
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(mock_error, color='#d64545', lw=2)
ax.axvline(int(np.argmin(mock_error)), color='#2e8b57', ls='--', label=f'lowest mock-exam error: tree {int(np.argmin(mock_error))}')
ax.set_xlabel('number of trees on the team'); ax.set_ylabel('mock-exam error (lower is better)')
ax.set_title('The coach watches the mock exam and stops when it stops improving'); ax.legend(); plt.show()

**How to read it:** the red line is the team's **error on the mock exam** after each tree (lower is better). It falls fast, then flattens. The dashed line marks the lowest point. The coach waited 50 more trees to be sure, then stopped.

**Stop and notice:**
- We allowed 3,000 trees. The coach stopped at about **350**, by itself, in about a second.
- Test AUC ≈ **0.945**: **boosting now beats the forest** (0.942).
- We **never looked at the test bookings** to choose the number of trees. That is the honest way.

**Early stopping is the one habit to remember from tonight.** With it, you can always set `max_iter` high and let the data decide.

## 6. Big steps or small steps? The learning rate

Remember `STEP = 0.3` in our by-hand team: we only added **30%** of each trainee's correction. That number is the **learning rate**, the second dial that matters.

Think of **golf**. You are far from the hole.

- **A big learning rate** is hitting every shot at full power. You get close fast, then you **fly past the hole**, and back, and past again.
- **A small learning rate** is a series of careful putts. Each one covers **part of the remaining distance**. More shots, but you **land close**.

Why not trust each trainee 100%? Because part of every trainee's correction is real pattern and part is **luck in the training data**. Small steps let later trainees fix earlier trainees' luck.

The next cell tries four learning rates. Early stopping picks the number of trees for each.

In [ ]:
rows = []
for rate in [1.0, 0.5, 0.1, 0.03]:
    start = time.time()
    m = HistGradientBoostingClassifier(learning_rate=rate, max_iter=3000, early_stopping=True,
                                       validation_fraction=0.15, n_iter_no_change=50, random_state=835).fit(X_tr, y_tr)
    rows.append({'learning_rate': rate, 'trees used': m.n_iter_,
                 'test AUC': roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]), 'seconds': time.time() - start})
pd.DataFrame(rows).round(3)

**Stop and notice:**
- **Big steps (1.0, 0.5): few trees, lower score.** The team rushed and overshot.
- **Small steps (0.1, 0.03): more trees, higher score.** Careful putts land closer.
- From 0.1 to 0.03 the score barely moves, but the team needs **more than twice as many trees**. Smaller is not always worth the wait.

**The recipe professionals use:** a learning rate between **0.03 and 0.1**, `max_iter` set high, and **early stopping** to choose the number of trees. The two dials work as a pair: **smaller steps need more trees.**

(There is a third dial, the **size of each tree**, `max_leaf_nodes`. You will turn it in Your Turn Q2.)

### 🧠 Quick check

Model A: learning rate 0.5, stopped at 70 trees. Model B: learning rate 0.05, stopped at 430 trees. Which one would you expect to score better on the exam, and what do you pay for it?

<details><summary>Click for the answer</summary>

**Model B**, usually. Small steps let each tree correct a little, so the team lands closer to the best fit. The price is **time**: six times as many trees to train, and six times as many to run for every new booking. On a table this size that is a second or two. On hundreds of millions of rows it is a real cost.
</details>

## 7. The famous cousins: XGBoost and LightGBM

You will see these two names in every job ad and every Kaggle competition.

| Name | Who made it | What to know |
|---|---|---|
| **XGBoost** | Tianqi Chen (2014) | the one that made boosting famous by winning Kaggle competitions |
| **LightGBM** | Microsoft (2017) | built for speed on very large tables; popularized the "Hist" bucket trick |
| **HistGradientBoosting** | scikit-learn | the same family, already inside the library we use every week |

**They are the same idea** you built by hand in Step 2: miss, small tree, add, repeat. They differ in speed, in details, and in the names of the dials (`max_iter` here is `n_estimators` there). Learn one and you can drive all three.

The next cell is **optional**. Both libraries come pre-installed on Colab. Same bookings, same learning rate, early stopping on a mock exam.

In [ ]:
# OPTIONAL — XGBoost and LightGBM on the same bookings (both are pre-installed on Colab; elsewhere: pip install xgboost lightgbm)
try:
    import warnings; warnings.filterwarnings('ignore')      # hide version notices from the two libraries
    import xgboost as xgb, lightgbm as lgb
    X_fit, X_mock, y_fit, y_mock = train_test_split(X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=835)   # our own mock exam

    xg = xgb.XGBClassifier(n_estimators=3000, learning_rate=0.1, early_stopping_rounds=50, random_state=835)
    xg.fit(X_fit, y_fit, eval_set=[(X_mock, y_mock)], verbose=False)
    print(f'XGBoost:   stopped at {xg.best_iteration:4d} trees'); score('XGBoost + early stopping', xg)

    lg = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.1, verbose=-1, random_state=835)
    lg.fit(X_fit, y_fit, eval_set=[(X_mock, y_mock)], callbacks=[lgb.early_stopping(50, verbose=False)])
    print(f'LightGBM:  stopped at {lg.best_iteration_:4d} trees'); score('LightGBM + early stopping', lg)
except ImportError:
    print('xgboost / lightgbm are not installed here. In Colab they are. This cell is optional: carry on.')

**Stop and notice:** three libraries, three teams of programmers, and the scores land **within a hair of each other**. The brand matters much less than the habits: **no leaks, a clean prep station, early stopping.**

## 8. What is new in 2026: models that have "read" millions of tables

Every model in this course so far **starts from zero**: it knows nothing until it sees *your* bookings.

ChatGPT does not start from zero. It read a large part of the internet first. That is a **foundation model**: trained once on a huge amount of data, then reused for many tasks.

Since 2025 the same idea works for **tables**. Models such as **TabPFN** and **TabICL** were trained in advance on **millions of made-up tables**. You hand one your training bookings and it predicts the test bookings **immediately**: no trees to grow, no dials to turn.

| | 🚀 Gradient boosting | 🧠 Tabular foundation model |
|---|---|---|
| Learns from | your table only | millions of practice tables first, then yours |
| Tuning | learning rate, trees, early stopping | **none** |
| Best at | big tables, fast answers, runs on any laptop | **small and medium tables**; the newest (TabPFN-3.5, Sept 2026) is recommended by its makers for up to a million rows |
| Watch out for | needs the stopping rule | slower to score new rows, wants a GPU, **check the license** before commercial use |

**What to take away:** on public benchmarks these new models now often match or beat a carefully tuned boosting model **with zero tuning**. This field changes every few months. Where speed, cost and simplicity matter (no GPU, no license questions, runs anywhere), **boosting is still the workhorse**. A good analyst in 2026 tries both.

The next cell is **optional** and needs an install (about a minute in Colab; a GPU runtime makes it much faster). It trains both models on the **same 5,000 bookings** and grades both on the **same 1,500 test bookings**.

In [ ]:
# OPTIONAL — a tabular foundation model, zero tuning. In Colab, run this line first in its own cell:   !pip install -q tabicl
try:
    from tabicl import TabICLClassifier
    small_X, small_y = X_tr.iloc[:5000], y_tr.iloc[:5000]             # 5,000 training bookings
    exam_X, exam_y = X_te.iloc[:1500], y_te.iloc[:1500]               # 1,500 test bookings (to keep it quick)

    small_boost = HistGradientBoostingClassifier(early_stopping=True, max_iter=3000, n_iter_no_change=50, random_state=835).fit(small_X, small_y)
    print(f'Gradient boosting on 5,000 bookings:         test AUC {roc_auc_score(exam_y, small_boost.predict_proba(exam_X)[:, 1]):.3f}')

    start = time.time()
    tfm = TabICLClassifier().fit(small_X, small_y)                    # no dials at all
    print(f'TabICL (foundation model) on 5,000 bookings: test AUC {roc_auc_score(exam_y, tfm.predict_proba(exam_X)[:, 1]):.3f}   ({time.time() - start:.0f} seconds)')
except ImportError:
    print('tabicl is not installed. This cell is optional: in Colab run  !pip install -q tabicl  and try again, or carry on.')
except Exception as problem:
    print('TabICL did not run here:', problem, '\nThis cell is optional: carry on.')

## The leaderboard

Everything from tonight, next to last week's best, on the same 6,000 test bookings.

In [ ]:
board = pd.DataFrame(leaderboard).drop_duplicates('model', keep='last').set_index('model')

colors = ['#999999' if m.startswith('Last week') else '#2e8b57' if m == board['test AUC'].idxmax() else '#5b6bd8' for m in board.index]
fig, ax = plt.subplots(figsize=(8, 0.55 * len(board) + 1))
ax.barh(board.index, board['test AUC'], color=colors)
for row, auc in enumerate(board['test AUC']):
    ax.text(auc + 0.002, row, f'{auc:.3f}', va='center')
ax.invert_yaxis(); ax.set_xlim(0.85, 0.96); ax.set_xlabel('test AUC (higher is better) · note: the axis starts at 0.85')
ax.set_title('Tonight: from a forest to a boosted team'); plt.show()

**Stop and notice (an honest reading):** boosting beats the forest here by about **0.003**. That is a real gain, but a small one. The big jump was last week, from one tree to a crowd. In practice the choice between a forest and boosting is rarely about the last decimal:

### Forest or boosting? When to use which

| | 🌲 **Random forest** | 🚀 **Gradient boosting** |
|---|---|---|
| Accuracy on tables | very good with no tuning (≈ 0.942 here) | **usually a bit better** when tuned (≈ 0.945 here) |
| More trees? | never hurts | **can memorize**: use early stopping |
| Dials that matter | almost none | learning rate, number of trees, tree size |
| Speed on big tables | slower | **very fast** (the bucket trick) |
| Blanks in the data | fill them first | **handles them itself** |
| Use it when… | you want a strong, safe model in one line | you want the **best score** and can spend ten minutes tuning |

Neither needs scaling or one-hot. Both are explained the same way as last week: **permutation importance** and **partial dependence** work on any model.

## 9. Your turn

Each question has a starter cell. Fill in the line marked `# ← your turn`, run it, then answer in words. Work with a neighbor if you like.

1. **The learning-rate dial.** Try two more learning rates. How do the number of trees and the score change?
2. **The size of each trainee.** `max_leaf_nodes` sets how many final boxes each small tree may have. Is bigger better?
3. **All the bookings.** Train the forest and the boosted team on all 83,573 bookings. Who wins, and by how much?
4. **Challenge (optional).** Force the model to respect business sense: *a longer lead time may never lower the chance of cancellation*.

**Q1 · The learning-rate dial.** Add two more learning rates to the list (for example 0.3 and 0.01).

In [ ]:
# Q1 · learning rate, with early stopping choosing the number of trees
for rate in [0.1]:          # ← your turn: add more values, e.g. [0.3, 0.1, 0.01]
    m = HistGradientBoostingClassifier(learning_rate=rate, max_iter=3000, early_stopping=True,
                                       validation_fraction=0.15, n_iter_no_change=50, random_state=835).fit(X_tr, y_tr)
    print(f'learning_rate {rate:<5}: stopped at {m.n_iter_:4d} trees   test AUC {roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]):.3f}')

Your answer to Q1 (double-click to edit): *with a learning rate of \_\_\_ the model used \_\_\_ trees and scored \_\_\_. Compared with 0.1, that is better / worse / about the same, because…* (Hint: with 0.01, did the model reach the 3,000-tree limit before the coach stopped it?)

**Q2 · The size of each trainee.** The default is 31 leaves per tree. Try smaller and bigger trainees.

In [ ]:
# Q2 · how big should each small tree be?
for leaves in [3, 31]:      # ← your turn: add more values, e.g. [3, 7, 15, 31, 63, 127]
    m = HistGradientBoostingClassifier(max_leaf_nodes=leaves, max_iter=3000, early_stopping=True,
                                       validation_fraction=0.15, n_iter_no_change=50, random_state=835).fit(X_tr, y_tr)
    print(f'max_leaf_nodes {leaves:3d}: stopped at {m.n_iter_:4d} trees   test AUC {roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]):.3f}')

Your answer to Q2: *the best size was \_\_\_ leaves (AUC \_\_\_). Tiny trainees (3 leaves) needed \_\_\_ trees. Very big trainees needed fewer trees but…*

**Q3 · All the bookings.** In class we used 30,000 bookings for speed. The starter prepares all 83,573. Two lines to finish: fit each model.

In [ ]:
# Q3 · the forest and the boosted team on ALL the bookings (about 67,000 to train, 17,000 as the exam)
X_all = raw.drop(columns=['is_canceled'] + LEAKS); y_all = raw['is_canceled']
A_raw, B_raw, y_A, y_B = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=835)
prep_all = tree_prep(X_all)
A = prep_all.fit_transform(A_raw); B = prep_all.transform(B_raw)

big_forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=835)
big_boost = HistGradientBoostingClassifier(max_iter=3000, early_stopping=True, validation_fraction=0.15, n_iter_no_change=50, random_state=835)

# ← your turn: fit both models on A, y_A  (for example: big_forest.fit(A, y_A)), then run the cell
for name, m in [('forest', big_forest), ('boosting', big_boost)]:
    if hasattr(m, 'classes_'):      # only grade a model that has been fitted
        print(f'{name:9s} test AUC {roc_auc_score(y_B, m.predict_proba(B)[:, 1]):.3f}')
    else:
        print(f'{name:9s} not fitted yet')

Your answer to Q3: *on all the bookings the forest scores \_\_\_ and boosting scores \_\_\_. Compared with 30,000 bookings, both models got better / worse, because… Which one trained faster?*

**Q4 · Challenge (optional): business sense as a rule.** A revenue manager may say: *"I will not accept a model in which booking **earlier** makes a guest look **safer**. That makes no sense to me."* Boosting can promise that. `monotonic_cst` takes one number per column: **1** = "the score may only go up as this column goes up", **0** = no rule.

In [ ]:
# Q4 · force: longer lead time can never LOWER the predicted chance of cancellation
rule = [1 if column == 'lead_time' else 0 for column in X_tr.columns]      # ← your turn: also try a rule for another column
m = HistGradientBoostingClassifier(monotonic_cst=rule, max_iter=3000, early_stopping=True,
                                   validation_fraction=0.15, n_iter_no_change=50, random_state=835).fit(X_tr, y_tr)
print(f'with the lead-time rule: test AUC {roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]):.3f}   (without the rule: {roc_auc_score(y_te, boost_es.predict_proba(X_te)[:, 1]):.3f})')

Your answer to Q4: *the rule cost about \_\_\_ of AUC. A manager might still want it because…* (Hint: think about explaining one strange prediction to a hotel owner, or to a regulator in lending.)

## What we learned tonight

- **Boosting builds a team one small tree at a time.** Each new tree studies the **miss** (what really happened minus the team's score) and corrects a little of it: **miss, small tree, add, repeat.** A forest is a committee that votes; boosting is a relay team.
- It is called **gradient** boosting because the miss tells every score **which way is downhill**. Each tree is one small step down the hill of error: the same idea that trains neural networks.
- Unlike a forest, **a boosted model can have too many trees.** **Early stopping** is the automatic brake: hold back a mock exam, and stop when it stops improving. Set `max_iter` high and let the data decide.
- The **learning rate** is the size of each step. **Small steps (0.03 to 0.1) with more trees** beat big steps. The two dials work as a pair.
- **XGBoost, LightGBM and HistGradientBoosting are the same idea** with different brand names. On table data, boosting is the model to beat, and it usually edges out the forest.
- **New in 2026:** tabular foundation models (TabPFN, TabICL) have "read" millions of tables in advance. On benchmarks they often match or beat tuned boosting with no tuning at all. Try both.

**Tonight, Mon Nov 9, 11:59 PM:** the one-page competition memo on Canvas (what you tried, what worked, what leaked, and one thing you would do with two more weeks). **HW3** (the threshold on your credit model + the Ames regression pipeline) is due **Mon Nov 16, 11:59 PM**. **Project Milestone 2** (a baseline model, scored with the metric your decision needs) is due **Mon Nov 23**.

**Next week:** *Trust, but verify.* How to tune a model honestly, explain one prediction to one customer, and say how sure the model is.